# Kapitel 7 — Die Rechnung eines neuronalen Netzes verstehen · Kommentierte Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch07_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Antwort beginnt mit der Überlegung auf Papier und zeigt dann ihre Entsprechung in Python.

**Inhalt.** Daten · Die Rechnungen der Lektionen 7.1 bis 7.3 · Optionaler Workshop (zwölf Nachrichten) · Aufgaben 7.1 bis 7.6

**Bevor Sie Jupyter öffnen:** Rechnen Sie zuerst von Hand den Vorwärtsdurchlauf von M7 (Lektion 7.1) und die Ableitung nach $w_{11}$ (Lektion 7.2). Die Zellen unten dienen dazu, diese Rechnungen zu **überprüfen**.

**Farbcode:** Indigo = Vorwärtsrechnung; Magenta gestrichelt = Weg des Gradienten; dicke Bernstein-Linie = untersuchtes Gewicht $w_{11}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farbcode des Buchs: Vorwärtsrechnung in Indigo, Gradient in Magenta (gestrichelt), untersuchtes Gewicht in dickem Bernstein
AVANT, RETOUR, POIDS_ETUDIE = '#3949ab', '#c2185b', '#f9a825'


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def perte(y, p):
    # logarithmischer Verlust einer Nachricht (Gleichung 5.6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)))


def aller(P, x):
    # Vorwärtsdurchlauf (Gleichungen 7.1 und 7.2): gibt ein Dictionary z1, a1, z2, a2, z, p zurück
    z1 = P['w11'] * x[0] + P['w12'] * x[1] + P['b1']
    z2 = P['w21'] * x[0] + P['w22'] * x[1] + P['b2']
    a1, a2 = sigmoide(z1), sigmoide(z2)
    z = P['v1'] * a1 + P['v2'] * a2 + P['c']
    return {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z': z, 'p': sigmoide(z)}


def dessiner(P, x, retour=False):
    # Das Schema des Buchs, vereinfacht: Indigo für den Vorwärtsdurchlauf, Magenta für den Rückweg, w11 in dickem Bernstein
    v = aller(P, x)
    pos = {'x1': (0, 3), 'x2': (0, 0), 'n1': (3, 3), 'n2': (3, 0), 's': (6, 1.5)}
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for a, b, nom in (('x1', 'n1', 'w11'), ('x2', 'n1', 'w12'), ('x1', 'n2', 'w21'), ('x2', 'n2', 'w22'),
                      ('n1', 's', 'v1'), ('n2', 's', 'v2')):
        (xa, ya), (xb, yb) = pos[a], pos[b]
        if nom == 'w11':
            ax.plot([xa, xb], [ya, yb], color=POIDS_ETUDIE, linewidth=7, zorder=1)
        ax.annotate('', xy=(xb - 0.35, yb + (ya - yb) * 0.12), xytext=(xa + 0.3, ya),
                    arrowprops=dict(arrowstyle='-|>', color=AVANT), zorder=2)
        t = 0.3 if a[0] == 'x' and b != ('n1' if a == 'x1' else 'n2') else 0.5
        ax.text(xa + t * (xb - xa), ya + t * (yb - ya) + 0.15, f"{nom} = {P[nom]:g}" + (' (untersuchtes Gewicht)' if nom == 'w11' else ''),
                color=AVANT, fontsize=8, ha='center', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85))
    textes = {'x1': f"x1 = {x[0]}", 'x2': f"x2 = {x[1]}",
              'n1': f"N1\nz1 = {v['z1']:.3f}\na1 = {v['a1']:.4f}", 'n2': f"N2\nz2 = {v['z2']:.3f}\na2 = {v['a2']:.4f}",
              's': f"Ausgabe\nz = {v['z']:.3f}\np̂ = {v['p']:.3f}"}
    for k, (cx, cy) in pos.items():
        ax.text(cx, cy, textes[k], ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='circle', facecolor='white', edgecolor=AVANT if k[0] != 'x' else 'grey'))
    if retour:
        for (x0, y0), (x1, y1), lab in (((6.6, 2.4), (3.5, 3.6), '② v1'), ((3.0, 3.9), (1.4, 3.4), '③ a1(1 − a1), ④ x1')):
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='-|>', color=RETOUR, linestyle='--', connectionstyle='arc3,rad=0.3'))
            ax.text((x0 + x1) / 2, max(y0, y1) + 0.35, lab, color=RETOUR, fontsize=8, ha='center')
        ax.text(6.6, 2.7, '① p̂ − y', color=RETOUR, fontsize=8)
    ax.set_xlim(-0.8, 7.4); ax.set_ylim(-0.8, 4.4); ax.axis('off'); plt.show()

## Daten

Die Kennungen D7.1, D7.E1 und D5.1 sind die des Buchs und der CSV-Dateien. Variablen- und Spaltennamen im Code bleiben französisch: `reseau` (Netz), `collegue` (Kollege), `douze` (die zwölf Nachrichten), `etiquette` (Label: `spam` / `legitime`), `liens` (Links), `mots_suspects` (verdächtige Wörter), `aller` (Vorwärtsdurchlauf), `retour` (Rückweg), `perte` (Verlust), `mise_a_jour` (Aktualisierung), `dérivée` / `nouvelle valeur` (Ableitung / neuer Wert).

In [ ]:
# Daten von Kapitel 7, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D7.1 — die neun Parameter des Netzes der Lektionen 7.1 bis 7.3
reseau = {'w11': 0.5, 'w12': 0.5, 'b1': -2, 'w21': 1, 'w22': -1, 'b2': 0, 'v1': 2, 'v2': -2, 'c': 0}

# D7.E1 — das Netz des Kollegen (Aufgaben 7.1 und 7.2)
collegue = {'w11': 1, 'w12': 1, 'b1': -4, 'w21': -1, 'w22': 2, 'b2': -1, 'v1': 2, 'v2': 1, 'c': -2}

# D5.1 — die zwölf Nachrichten aus Kapitel 5: Label, Links, verdächtige Wörter
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

M7 = (2, 3)   # Nachricht M7: 2 Links, 3 verdächtige Wörter, Spam (y = 1)
M3 = (4, 0)   # Newsletter M3: 4 Links, 0 verdächtige Wörter, legitim (y = 0)

## Die Rechnungen der Lektionen, in Python

Diese Zellen wiederholen die Rechnungen der Lektionen 7.1 bis 7.3.

In [ ]:
# Lektion 7.1 — Vorwärtsdurchlauf der Nachricht M7 (Gleichungen 7.1 und 7.2)
v = aller(reseau, M7)
print(f"N1: z1 = {v['z1']:.3f}, a1 = {v['a1']:.4f}")
print(f"N2: z2 = {v['z2']:.3f}, a2 = {v['a2']:.4f}")
print(f"Ausgabe: z = {v['z']:.3f}, p̂ = {v['p']:.3f}, Verlust = {perte(1, v['p']):.3f}")
print(f"M3 (Newsletter): a2 = {aller(reseau, M3)['a2']:.3f}, p̂ = {aller(reseau, M3)['p']:.3f}")
dessiner(reseau, M7)

In [ ]:
# Lektion 7.2 — die Kettenregel für w11 (Gleichung 7.3), Faktor für Faktor
f1 = v['p'] - 1                  # ① Verlust ← z: p̂ − y
f2 = reseau['v1']                # ② z ← a1: v1
f3 = v['a1'] * (1 - v['a1'])     # ③ a1 ← z1: Ableitung der Sigmoidfunktion, mit der AKTIVIERUNG a1
f4 = M7[0]                       # ④ z1 ← w11: x1
d_w11 = f1 * f2 * f3 * f4
print(f"① {f1:.3f}  ② {f2:g}  ③ {f3:.4f}  ④ {f4}  → Ableitung = {d_w11:.3f}")

# Kontrolle durch eine kleine numerische Störung (wie die Sekanten aus Lektion 3.3)
def perte_w11(w):
    return perte(1, aller(dict(reseau, w11=w), M7)['p'])
h = 0.0001
print(f"numerische Ableitung: ({perte_w11(0.5 + h):.6f} − {perte_w11(0.5 - h):.6f}) / (2h) = "
      f"{(perte_w11(0.5 + h) - perte_w11(0.5 - h)) / (2 * h):.4f}")

# Eine Aktualisierung von Hand, η = 0,5, dann wird der Vorwärtsdurchlauf wiederholt
eta = 0.5
w11_nouveau = round(0.5 - eta * d_w11, 3)   # auf Tausendstel gerundet, wie im Buch: 0,655
v_n = aller(dict(reseau, w11=w11_nouveau), M7)
print(f"w11: 0,5 → {w11_nouveau:.3f}; z1 = {v_n['z1']:.2f}, a1 = {v_n['a1']:.4f}, z = {v_n['z']:.3f}, "
      f"p̂ = {v_n['p']:.3f}, Verlust = {perte(1, v_n['p']):.3f}")
dessiner(reseau, M7, retour=True)

ws = np.linspace(-1, 3, 200)
plt.figure(figsize=(6, 3))
plt.plot(ws, [perte_w11(w) for w in ws], color='dimgrey', label='Verlust von M7 in Abhängigkeit von w11')
plt.plot(ws[25:126], perte_w11(0.5) + d_w11 * (ws[25:126] - 0.5), '--', color=RETOUR, label='Tangente bei 0,5')
plt.scatter([0.5, w11_nouveau], [perte_w11(0.5), perte_w11(w11_nouveau)], color=['black', AVANT], zorder=3)
plt.xlabel('w11'); plt.ylabel('Verlust von M7'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lektion 7.3 — Backpropagation: die neun Ableitungen (Gleichung 7.4), dann eine Aktualisierung ALLER Parameter
def retour(P, x, y):
    v = aller(P, x)
    d = v['p'] - y                                   # Fehlersignal der Ausgabe
    d1 = d * P['v1'] * v['a1'] * (1 - v['a1'])       # Fehlersignal von N1
    d2 = d * P['v2'] * v['a2'] * (1 - v['a2'])       # Fehlersignal von N2
    return {'w11': d1 * x[0], 'w12': d1 * x[1], 'b1': d1, 'w21': d2 * x[0], 'w22': d2 * x[1], 'b2': d2,
            'v1': d * v['a1'], 'v2': d * v['a2'], 'c': d}


def mise_a_jour(P, g, eta):
    return {k: P[k] - eta * g[k] for k in P}        # alle Ableitungen werden VOR dem Ändern berechnet


g = retour(reseau, M7, 1)
nouveau = mise_a_jour(reseau, g, 0.5)
print(pd.DataFrame({'valeur': reseau, 'dérivée': g, 'nouvelle valeur': nouveau}).round(3))
print(f"M7: p̂ = {aller(nouveau, M7)['p']:.3f}, Verlust = {perte(1, aller(nouveau, M7)['p']):.3f}")
print(f"M3 vorher: p̂ = {aller(reseau, M3)['p']:.3f}, Verlust = {perte(0, aller(reseau, M3)['p']):.3f}; "
      f"nachher: p̂ = {aller(nouveau, M3)['p']:.3f}, Verlust = {perte(0, aller(nouveau, M3)['p']):.3f}")

## Optionaler Workshop — das Netz auf den zwölf Nachrichten trainieren

1000 Schritte Gradientenabstieg ($\eta = 0{,}5$), ausgehend von den Gewichten aus Lektion 7.1; bei jedem Schritt werden die Ableitungen der zwölf Nachrichten aus Kapitel 5 gemittelt. **Alle Messungen unten beziehen sich auf diese zwölf Nachrichten, die zum Training dienen**: Sie sagen nichts darüber, wie sich das Netz bei neuen Nachrichten verhält (Lektion 6.3).

In [ ]:
# Optionaler Workshop: Training auf den zwölf Nachrichten (gemessen auf den Trainingsdaten)
X12 = list(zip(douze['liens'], douze['mots_suspects']))
y12 = [1 if e == 'spam' else 0 for e in douze['etiquette']]


def perte_12(P):
    return np.mean([perte(y, aller(P, x)['p']) for x, y in zip(X12, y12)])


def gradient_12(P):
    gs = [retour(P, x, y) for x, y in zip(X12, y12)]
    return {k: np.mean([g[k] for g in gs]) for k in P}     # Ableitung des mittleren Verlusts = Mittelwert der Ableitungen


P_entraine, suivi = dict(reseau), [perte_12(reseau)]
for _ in range(1000):
    P_entraine = mise_a_jour(P_entraine, gradient_12(P_entraine), 0.5)
    suivi.append(perte_12(P_entraine))
print(f"mittlerer Verlust auf den zwölf Trainingsnachrichten: vorher {suivi[0]:.3f}, nachher {suivi[-1]:.3f}")
for nom, x in (('M7', M7), ('M3', M3)):
    print(f"{nom}: p̂ vorher {aller(reseau, x)['p']:.3f}, nachher {aller(P_entraine, x)['p']:.3f}")
print('Fehler bei Schwelle 0,5 auf den zwölf: vorher', sum((aller(reseau, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)),
      ', nachher', sum((aller(P_entraine, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)))
print('Gewichte nach dem Training:', {k: round(float(v), 2) for k, v in P_entraine.items()})
plt.figure(figsize=(6, 3)); plt.plot(suivi, color=AVANT)
plt.xlabel('Schritt'); plt.ylabel("mittlerer Verlust (zwölf Trainingsnachrichten)"); plt.grid(alpha=0.4); plt.show()

## Aufgabe 7.1 — Das Netz eines Kollegen lesen

*Begriffe: Schichten, Gewichte und Bias (Lektion 7.1, Gleichungen 7.1 und 7.2).* Daten D7.E1: das Dictionary `collegue`.

### a) bis e) Kommentierte Lösung

**a)** Sechs Gewichte, in der verborgenen Schicht ($w_{11}$, $w_{12}$, $w_{21}$, $w_{22}$) und in der Ausgabe ($v_1$, $v_2$), dazu drei Bias-Werte ($b_1$, $b_2$, $c$): neun Parameter, wie beim Netz aus Lektion 7.1.

**b)** $w_{21} = -1$ verbindet die Links mit dem Neuron N2; $w_{12} = 1$ verbindet die verdächtigen Wörter mit dem Neuron N1.

**c)** Der Bias −4 gehört zu N1. Der Bias der Ausgabe ist $c = -2$.

**d)** Mit $x_1 = x_2 = 0$ sind die Produkte mit den Gewichten null: $z_1 = b_1 = -4$ und $z_2 = b_2 = -1$. Der Bias ist die gewichtete Summe einer „leeren“ Nachricht.

**e)** Hier berechnet N2 $-x_1 + 2x_2 - 1$: Der Wert wächst mit den verdächtigen Wörtern und sinkt mit den Links, und das Ausgabegewicht $v_2 = 1$ ist positiv: Steigt die Aktivierung, dann **steigt** die Spam-Wahrscheinlichkeit. Im Netz aus Lektion 7.1 berechnete N2 $x_1 - x_2$, und sein Ausgabegewicht war negativ: fast die umgekehrte Wirkung.

In [ ]:
nb_parametres = len(collegue)
vide = aller(collegue, (0, 0))

In [ ]:
# Kontrolle von a) und d)
if nb_parametres is None or vide is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    print(f"{nb_parametres} Parameter; leere Nachricht: z1 = {vide['z1']:g}, z2 = {vide['z2']:g} →",
          'richtig' if nb_parametres == 9 and vide['z1'] == -4 and vide['z2'] == -1 else 'bitte prüfen')

## Aufgabe 7.2 — Eine Ausgabe von Hand berechnen

*Begriffe: Vorwärtsdurchlauf (Gleichungen 7.1 und 7.2), Sigmoidfunktion (Gleichung 5.4), Verlust (Gleichung 5.6), Entscheidungsregel (Gleichung 5.1).*

### a) bis c) Kommentierte Lösung

**a)** M6: $z_1 = 1 \times 2 + 1 \times 2 - 4 = 0$, $a_1 = \sigma(0) = 0{,}5$; $z_2 = -1 \times 2 + 2 \times 2 - 1 = 1$, $a_2 = \sigma(1) \approx 0{,}7311$; $z = 2 \times 0{,}5 + 1 \times 0{,}7311 - 2 \approx -0{,}269$; $\hat p = \sigma(-0{,}269) \approx 0{,}433$.

**b)** $0{,}433 < 0{,}5$: M6 wird nicht markiert, und das ist richtig, denn die Nachricht ist legitim. Ihr Verlust beträgt $-\ln(1 - 0{,}433) \approx 0{,}568$.

**c)** M7: $z_1 = 2 + 3 - 4 = 1$, $a_1 \approx 0{,}7311$; $z_2 = -2 + 6 - 1 = 3$, $a_2 \approx 0{,}9526$; $z = 1{,}4621 + 0{,}9526 - 2 \approx 0{,}415$; $\hat p \approx 0{,}602$, Verlust $-\ln(0{,}602) \approx 0{,}507$. Das Netz aus Lektion 7.1 lässt M7 einen kleineren Verlust zahlen, 0,401: Es gibt der Nachricht eine Wahrscheinlichkeit näher bei 1.

In [ ]:
p_M6 = aller(collegue, (2, 2))['p']
perte_M6 = perte(0, p_M6)                          # M6 ist legitim: y = 0
perte_M7 = perte(1, aller(collegue, M7)['p'])      # M7 ist Spam: y = 1

In [ ]:
# Kontrolle von a) bis c)
if p_M6 is None or perte_M6 is None or perte_M7 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Zeilen.')
else:
    print(f"M6: p̂ = {p_M6:.3f}, markiert: {p_M6 >= 0.5}, Verlust = {perte_M6:.3f} →",
          'richtig' if abs(p_M6 - 0.4332) < 5e-4 and abs(perte_M6 - 0.5677) < 5e-4 else 'bitte prüfen')
    print(f"M7: Verlust {perte_M7:.3f} (Kollege) gegenüber {perte(1, aller(reseau, M7)['p']):.3f} (Lektion 7.1) →",
          'richtig' if abs(perte_M7 - 0.5071) < 5e-4 else 'bitte prüfen')
    dessiner(collegue, (2, 2))

## Aufgabe 7.3 — Eine Aktivierung deuten

*Begriffe: Rolle eines verborgenen Neurons (Lektion 7.1).* Netz aus Lektion 7.1, zwölf Nachrichten aus Kapitel 5.

### a) bis d) Kommentierte Lösung

**a)** $a_2$ übersteigt 0,8 bei M2 (0,881) und M3 (0,982): zwei legitime Nachrichten, mit Links und **ohne** verdächtige Wörter. $a_2$ ist genau 0,5, wenn $z_2 = x_1 - x_2 = 0$ gilt, also wenn die Nachricht ebenso viele Links wie verdächtige Wörter hat: M4, M6, M9.

**b)** N2 leuchtet auf (Aktivierung über 0,5), wenn eine Nachricht mehr Links als verdächtige Wörter hat, und umso stärker, je größer der Unterschied ist. Sein Ausgabegewicht $v_2 = -2$ senkt dann den Ausgabewert und damit die Spam-Wahrscheinlichkeit. Unter diesen sechs Nachrichten sind die beiden, bei denen N2 am stärksten aufleuchtet, legitim; aber sechs Nachrichten mit von Hand gewählten Gewichten reichen nicht, um zu sagen, dass N2 „Newsletter erkennt“: Das müsste man an vielen neuen Nachrichten prüfen.

**c)** M6: $z = 2 \times 0{,}5 - 2 \times 0{,}5 + 0 = 0$ genau, also $\hat p = 0{,}5$ genau. Mit der Regel „größer oder gleich“ wird M6 markiert: ein Fehlalarm, genau an der Schwelle. Das ist die Situation von M3 in Kapitel 5, dessen Rohwert genau 0 war.

**d)** Es ist N2: Mit $a_2 \approx 0{,}982$ zieht es etwa 1,96 vom Ausgabewert ab, und das Netz gibt M3 nur noch 0,276. Eine Aktivierung ist keine Spam-Wahrscheinlichkeit: Sie ist eine Zahl zwischen 0 und 1, die als Zutat für die Ausgabe dient. Nur $\hat p$ in der Ausgabe ist eine geschätzte Wahrscheinlichkeit.

In [ ]:
douze['a2'] = [aller(reseau, (l, m))['a2'] for l, m in zip(douze['liens'], douze['mots_suspects'])]
allumes = list(douze.loc[douze['a2'] > 0.8, 'message'])

In [ ]:
# Kontrolle von a) und c)
if allumes is None or douze['a2'].isna().any():
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    douze['a1'] = [aller(reseau, (l, m))['a1'] for l, m in zip(douze['liens'], douze['mots_suspects'])]
    douze['p̂'] = [aller(reseau, (l, m))['p'] for l, m in zip(douze['liens'], douze['mots_suspects'])]
    print(douze.round(3).to_string(index=False))
    print('a2 > 0,8:', allumes, '→', 'richtig' if allumes == ['M2', 'M3'] else 'bitte prüfen')
    print(f"M6: z = {aller(reseau, (2, 2))['z']}, p̂ = {aller(reseau, (2, 2))['p']} → markiert bei Schwelle 0,5: {aller(reseau, (2, 2))['p'] >= 0.5}")

## Aufgabe 7.4 — Eine Ableitung in Jupyter ergänzen

*Begriffe: Kettenregel (Gleichung 7.3), numerische Kontrolle (Lektion 7.2).* Das Gewicht $w_{22}$ verbindet die verdächtigen Wörter mit dem Neuron N2.

### a) und b) Kommentierte Lösung

**a)** ① $\hat p - y \approx -0{,}3303$; ② $v_2 = -2$; ③ $a_2(1 - a_2) = 0{,}2689 \times 0{,}7311 \approx 0{,}1966$; ④ $x_2 = 3$. Produkt: $-0{,}3303 \times (-2) \times 0{,}1966 \times 3 \approx 0{,}390$.

**b)** Das Vorzeichen ist positiv: Eine Erhöhung von $w_{22}$ lässt den Verlust **steigen**. Verfolgen Sie die Vorzeichen: Steigt $w_{22}$, so steigt $z_2$ (weil $x_2 = 3 > 0$) und damit $a_2$; da $v_2 < 0$, sinkt $z$; $\hat p$ sinkt, obwohl M7 ein Spam ist, dessen Wahrscheinlichkeit schon zu niedrig ist. Zwei negative Faktoren, ① und ②, ergeben ein positives Produkt.

In [ ]:
def derivee_w22(x, y):
    v = aller(reseau, x)
    return (v['p'] - y) * reseau['v2'] * v['a2'] * (1 - v['a2']) * x[1]   # ① × ② × ③ × ④

In [ ]:
# Kontrolle von c): Kettenregel gegen numerische Störung
if derivee_w22(M7, 1) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    h = 0.0001
    num = (perte(1, aller(dict(reseau, w22=-1 + h), M7)['p']) - perte(1, aller(dict(reseau, w22=-1 - h), M7)['p'])) / (2 * h)
    print(f"Kettenregel: {derivee_w22(M7, 1):.4f} →", 'richtig' if abs(derivee_w22(M7, 1) - 0.3896) < 5e-4 else 'bitte prüfen')
    print(f"numerische Ableitung: {num:.4f} → Abweichung {abs(num - derivee_w22(M7, 1)):.1e} →",
          'richtig' if abs(num - derivee_w22(M7, 1)) < 1e-6 else 'bitte prüfen')

## Aufgabe 7.5 — Vorhersagen, dann prüfen: die Wirkung einer Gewichtsänderung

*Begriffe: Ableitung und Näherung (Lektionen 3.3 und 7.2).*

### a) Kommentierte Lösung

Nur mit der Ableitung: Verlust $\approx 0{,}401 - 0{,}310 \times \Delta w$. Für $\Delta w = 0{,}1$: $0{,}401 - 0{,}031 = 0{,}370$. Für $\Delta w = 2$: $0{,}401 - 0{,}620 \approx -0{,}22$, ein **negativer** Verlust, was unmöglich ist.

In [ ]:
def perte_apres(dw):
    return perte(1, aller(dict(reseau, w11=0.5 + dw), M7)['p'])   # der ganze Vorwärtsdurchlauf wird wiederholt

In [ ]:
# Kontrolle von b)
if perte_apres(0.1) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    L0 = perte(1, aller(reseau, M7)['p'])
    for dw in (0.1, 2):
        print(f"dw = {dw}: vorhergesagt {L0 + d_w11 * dw:.3f}, exakt {perte_apres(dw):.3f}")
    print('→', 'richtig' if abs(perte_apres(0.1) - 0.3716) < 5e-4 and abs(perte_apres(2) - 0.2126) < 5e-4 else 'bitte prüfen')

### b) und c) Kommentierte Lösung

Exakte Verluste: 0,372 für $w_{11} = 0{,}6$, sehr nah an den vorhergesagten 0,370; 0,213 für $w_{11} = 2{,}5$, sehr weit weg von den vorhergesagten −0,22. Die Ableitung beschreibt die Kurve **nahe** am Ausgangspunkt: Bei einem kleinen Schritt liegt die Tangente eng an der Kurve, danach entfernt sie sich (Abbildung in Lektion 7.2). Bei einem großen Schritt flacht der Verlust ab und kann nicht negativ werden. Deshalb bleibt die Lernrate klein: Jeder Schritt muss in dem Bereich bleiben, in dem die Ableitung stimmt, auch wenn man dafür viele Schritte braucht.

## Aufgabe 7.6 — Herausforderung: mehrere Gewichte auf zwei Nachrichten aktualisieren

*Begriffe: Backpropagation (Gleichung 7.4), mittlerer Verlust (Gleichung 5.7), Gradientenabstieg (Gleichung 3.7).* Nachrichten M7 (Spam) und M3 (legitim).

### a) bis c) Kommentierte Lösung

**a)** Anfangsverluste: 0,401 für M7 und $-\ln(1 - 0{,}2761) \approx 0{,}323$ für M3; Mittelwert $\approx 0{,}362$.

**b)** $\delta_{M7} = 0{,}6697 - 1 \approx -0{,}330$ und $\delta_{M3} = 0{,}2761 - 0 = 0{,}2761$. Ableitungen des mittleren Verlusts: $\partial L/\partial v_1 = (-0{,}330 \times 0{,}6225 + 0{,}2761 \times 0{,}5)/2 \approx -0{,}034$; $\partial L/\partial v_2 = (-0{,}330 \times 0{,}2689 + 0{,}2761 \times 0{,}9820)/2 \approx 0{,}091$; $\partial L/\partial c = (-0{,}330 + 0{,}2761)/2 \approx -0{,}027$.

**c)** Mit $\eta = 1$: $v_1 \approx 2{,}034$, $v_2 \approx -2{,}091$, $c \approx 0{,}027$. Die verborgenen Aktivierungen ändern sich nicht. Neue Verluste: etwa 0,393 für M7 und 0,311 für M3, Mittelwert 0,352. Der mittlere Verlust sinkt, und **jeder** der beiden Verluste ebenfalls: Anders als bei der Aktualisierung nur mit M7 (Lektion 7.3) wird M3 nicht geopfert, weil seine Ableitung in den Mittelwert eingeht.

In [ ]:
def gradient_moyen(P):
    g7, g3 = retour(P, M7, 1), retour(P, M3, 0)
    return {k: (g7[k] + g3[k]) / 2 for k in P}    # Ableitung eines Mittelwerts = Mittelwert der Ableitungen

In [ ]:
# Kontrolle von c), dann d): die neun Parameter, ein Schritt, dann 100 Schritte
def perte_moyenne(P):
    return (perte(1, aller(P, M7)['p']) + perte(0, aller(P, M3)['p'])) / 2

if gradient_moyen(reseau) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    g = gradient_moyen(reseau)
    sortie_seule = dict(reseau, v1=reseau['v1'] - g['v1'], v2=reseau['v2'] - g['v2'], c=reseau['c'] - g['c'])
    print(f"Start: mittlerer Verlust {perte_moyenne(reseau):.3f}")
    print(f"c) nur v1, v2, c: M7 {perte(1, aller(sortie_seule, M7)['p']):.3f}, "
          f"M3 {perte(0, aller(sortie_seule, M3)['p']):.3f}, Mittelwert {perte_moyenne(sortie_seule):.3f} →",
          'richtig' if abs(perte_moyenne(sortie_seule) - 0.3519) < 5e-4 else 'bitte prüfen')
    neuf = mise_a_jour(reseau, g, 1)
    print(f"d) die neun Parameter, ein Schritt: Mittelwert {perte_moyenne(neuf):.3f} →",
          'richtig' if abs(perte_moyenne(neuf) - 0.2584) < 5e-4 else 'bitte prüfen')
    P, suivi = dict(reseau), []
    for _ in range(100):
        P = mise_a_jour(P, gradient_moyen(P), 1)
        suivi.append(perte_moyenne(P))
    print(f"   100 Schritte: p̂(M7) = {aller(P, M7)['p']:.3f}, p̂(M3) = {aller(P, M3)['p']:.3f}, mittlerer Verlust {suivi[-1]:.3f}")
    plt.figure(figsize=(6, 3)); plt.plot(range(1, 101), suivi, color=AVANT)
    plt.xlabel('Schritt'); plt.ylabel('mittlerer Verlust (M7 und M3)'); plt.grid(alpha=0.4); plt.show()

### d) Kommentierte Lösung

Ein Schritt auf allen neun Parametern senkt den mittleren Verlust von 0,362 auf 0,258: viel stärker als die Ausgabeschicht allein (0,352), weil sich auch die verborgenen Neuronen anpassen. Nach 100 Schritten gibt das Netz M7 den Wert 0,982 und M3 den Wert 0,018: mittlerer Verlust 0,018. Dieser Erfolg beweist nichts für später: Mit neun Parametern für zwei Nachrichten kann das Netz sie auswendig lernen, wie Filter B in Lektion 6.3. Nur Nachrichten, die es nicht gesehen hat, würden zeigen, was es taugt.